## NetworkPolicy — the built-in firewall

Out of the box the cluster is **open** — every pod reaches every pod, in every namespace. Orchestration first, security opt-in. **NetworkPolicy** is the opt-in: a namespaced object that, for pods matching a selector, controls **ingress** (who can reach them) and **egress** (where they can go).

```yaml
kind: NetworkPolicy
spec:
  podSelector: { matchLabels: { app: web } }
  policyTypes: [Ingress, Egress]
  ingress:
    - from:
        - podSelector: { matchLabels: { role: frontend } }
      ports: [{ protocol: TCP, port: 80 }]
  egress:
    - to: [{ podSelector: { matchLabels: { app: postgres } } }]
      ports: [{ protocol: TCP, port: 5432 }]
```

Three semantics to internalise:

1. **Default-deny once any policy applies.** No policy selects a pod → unrestricted (cluster allow-all). The moment **one** policy selects it, that pod becomes deny-by-default for the listed `policyTypes`, and only explicit allows pass. So `policyTypes: [Ingress]` with empty `ingress: []` = **deny-all-ingress**.
2. **Multiple policies are additive (a union).** Traffic is allowed if **any** policy allows it — no deny rules, no precedence. You build coverage from a default-deny plus targeted allows.
3. **Enforcement is the CNI's job.** The API server stores the object; the CNI drops/allows packets. **Not every CNI implements it** — Calico/Cilium yes, **Flannel and `kindnet` no**. Apply a policy on a non-enforcing CNI and it silently does nothing.

**`from`/`to` gotcha:** entries in the list are **OR'd**; selectors *within* one entry are **AND'd** — `namespaceSelector` + `podSelector` in the same entry means "frontend pods *in* team-a namespaces," not "either." Also supports `ipBlock` (CIDR + `except`). The canonical recipe is **default-deny-all** (`podSelector: {}`, both policyTypes) then narrow allows — remembering to **re-allow DNS to CoreDNS** on UDP/TCP 53. On our map this is the **NetworkPolicy** chip filtering the arrows into and out of the **Pods**.